# PROJECT STAGE

Current stage: Research exploration

Code may later be migrated to src modules once stabilized.

# 1. Metadata

In [ ]:
# ============================================================
# NOTEBOOK METADATA
# ============================================================

NOTEBOOK_NAME = "02_asset_characterization_Volatility"
AUTHOR = "Juan Manuel Giacame"
CREATED = "2026-03-11"
UPDATED = "2026-03-12"
PROJECT = "macro-market-analysis"
STAGE = "Research"
VERSION = "0.1.0"
GIT_HASH = ""  # completar con commit hash si se usa git
EXPERIMENT_ID = f"{NOTEBOOK_NAME}_{CREATED}_{VERSION}"

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
PURPOSE = """
Compute volatility features for each asset in the universe.

This notebook calculates time-series volatility metrics using
rolling return aggregation across multiple horizons.

The resulting features are stored per asset in:

data/features/asset/volatility/{asset}.parquet
"""

# ------------------------------------------------------------
# INPUT DATASETS
# ------------------------------------------------------------
INPUT_DATASETS = [
    "data/processed/prices.parquet"
    "data/processed/returns.parquet"
]

# ------------------------------------------------------------
# OUTPUT DATASETS
# ------------------------------------------------------------
OUTPUT_DATASETS = [
    "data/features/asset/volatility/{asset}.parquet"
]

# ------------------------------------------------------------
# FEATURE FAMILY
# ------------------------------------------------------------
FEATURE_FAMILY = "Volatility"

# ------------------------------------------------------------
# FEATURE DESCRIPTION
# ------------------------------------------------------------
FEATURE_DESCRIPTION = """
Volatility features measure the variation of asset returns
over multiple historical horizons.

These features capture how unstable or stable an asset has been,
helping to quantify risk, detect regime changes, and inform
systematic allocation or hedging strategies.
"""
# ------------------------------------------------------------
# ASSETS UNIVERSE
# ------------------------------------------------------------
ASSETS_UNIVERSE = "All assets in data/processed/returns.parquet"

# ------------------------------------------------------------
# DEPENDENCIES
# ------------------------------------------------------------
DEPENDENCIES = ["pandas >= 2.0", "numpy", "plotly >= 5.0", "pathlib"]

# ------------------------------------------------------------
# NOTES
# ------------------------------------------------------------
NOTES = """
Features are computed independently per asset.

Cross-sectional transformations such as:
    - rankings
    - z-scores
    - dispersion
are handled later in:

03_systemic_features.ipynb
"""

# 2. Imports

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

# -----------------------------
# Standard library
# -----------------------------
from pathlib import Path

# -----------------------------
# Data manipulation
# -----------------------------
import pandas as pd  # pandas >= 2.0
import numpy as np   # numpy >= 1.25

# Optional: display settings for research notebooks
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.options.mode.chained_assignment = None  # avoid SettingWithCopyWarning

# -----------------------------
# Parquet engine
# -----------------------------
import pyarrow  # pyarrow >= 12.0

# -----------------------------
# Visualization
# -----------------------------
import plotly.express as px  # plotly >= 5.0
import plotly.graph_objects as go

# -----------------------------
# Optional future imports from src/ once implemented
# -----------------------------
# from src.utils import panel_utils, plotting_utils
# from src.features.volatility import compute_volatility_features

# 3. Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# DATA PATHS
# ------------------------------------------------------------
DATA_DIR = Path("../../data")
PROCESSED_DATA_DIR = DATA_DIR / "processed"
FEATURES_DIR = DATA_DIR / "features"

# Input dataset for volatility calculations
PRICES_PATH = PROCESSED_DATA_DIR / "prices.parquet"
RETURNS_PATH = PROCESSED_DATA_DIR / "returns.parquet"
# ------------------------------------------------------------
# FEATURE FAMILY
# ------------------------------------------------------------
FEATURE_FAMILY = "Volatility"

# ------------------------------------------------------------
# FEATURE PARAMETERS
# ------------------------------------------------------------
LOOKBACK_WINDOWS = [21, 63, 126, 252]

# Smoothing windows for velocity/acceleration derivatives (VEL_S, ACC_S)
SMOOTH_WINDOWS = {
    21: 3,
    63: 3,
    126: 5,
    252: 5
}

VOL_SHORT_WINDOW = 21
VOL_MEDIUM_WINDOW = 63
VOL_LONG_WINDOW = 126

# ------------------------------------------------------------
# VOLATILITY OF VOLATILITY PARAMETERS
# ------------------------------------------------------------
VOV_CONFIG = {
    f"VOV_{VOL_SHORT_WINDOW}_{VOL_MEDIUM_WINDOW}": (VOL_SHORT_WINDOW, VOL_MEDIUM_WINDOW),
    f"VOV_{VOL_MEDIUM_WINDOW}_{VOL_LONG_WINDOW}": (VOL_MEDIUM_WINDOW, VOL_LONG_WINDOW),
    f"VOV_{VOL_SHORT_WINDOW}_{VOL_LONG_WINDOW}": (VOL_SHORT_WINDOW, VOL_LONG_WINDOW),
}
# ------------------------------------------------------------
# VOLATILITY REGIME PARAMETERS
# ------------------------------------------------------------
Z_SCORE_WINDOW = 252
PERCENTILE_WINDOW = 252

# Volatility expansion pairs
EXPANSION_PAIRS = [
    (VOL_SHORT_WINDOW, VOL_MEDIUM_WINDOW)
]

# ------------------------------------------------------------
# UNIVERSE CONFIGURATION
# ------------------------------------------------------------
# Will be dynamically extracted from df_prices after loading
ASSETS = None
UNIVERSE_SOURCE = "prices_columns"

# ------------------------------------------------------------
# OUTPUT CONFIGURATION
# ------------------------------------------------------------
EXPORT_FORMAT = "parquet"
OVERWRITE_EXISTING = True

# ------------------------------------------------------------
# RESEARCH OPTIONS
# ------------------------------------------------------------
# Sample assets for visualizations and sanity checks
SAMPLE_ASSETS_FOR_PLOTS = [
    "BTC",
    "SPY"
]

# ------------------------------------------------------------
# RANDOM SEED
# ------------------------------------------------------------
RANDOM_SEED = 42

# 4. Data Loading

In [ ]:
# ============================================================
# 4. DATA LOADING
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# Helper function: detect project root by searching for a marker folder
# ------------------------------------------------------------
def find_project_root(marker="data") -> Path:
    """
    Busca hacia arriba en la jerarquía de directorios hasta encontrar
    una carpeta que contenga `marker` y la devuelve como Path.
    """
    current_path = Path().resolve()
    for parent in [current_path] + list(current_path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError(f"Project root with '{marker}/' directory not found.")

project_root = find_project_root("data")
print("Project root detected at:", project_root)

# ------------------------------------------------------------
# Define data paths
# ------------------------------------------------------------
DATA_DIR = project_root / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
FEATURES_DIR = DATA_DIR / "features" / "asset" / "volatility"

PRICES_PATH = PROCESSED_DATA_DIR / "prices.parquet"
RETURNS_PATH = PROCESSED_DATA_DIR / "returns.parquet"

# ------------------------------------------------------------
# Verify files exist
# ------------------------------------------------------------
for path in [PRICES_PATH, RETURNS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required dataset not found at {path}")
print("All input datasets found ✔")

# ------------------------------------------------------------
# Load datasets
# ------------------------------------------------------------
df_prices = pd.read_parquet(PRICES_PATH)
df_returns = pd.read_parquet(RETURNS_PATH)

# Ensure datetime index
if "date" in df_prices.columns:
    df_prices["date"] = pd.to_datetime(df_prices["date"])
    df_prices.set_index("date", inplace=True)
df_prices.sort_index(inplace=True)

if "date" in df_returns.columns:
    df_returns["date"] = pd.to_datetime(df_returns["date"])
    df_returns.set_index("date", inplace=True)
df_returns.sort_index(inplace=True)

print(f"Prices dataset shape: {df_prices.shape}")
print(f"Returns dataset shape: {df_returns.shape}")

# ------------------------------------------------------------
# Infer asset universe
# ------------------------------------------------------------
ASSETS = df_prices.columns.tolist()
print(f"\nNumber of assets in universe: {len(ASSETS)}")
print("Sample assets:", ASSETS[:10])

# ------------------------------------------------------------
# Quick inspection
# ------------------------------------------------------------
display(df_prices.head())
display(df_returns.head())

# 5. Core Computations  

Volatility Feature Review

Volatility measures the magnitude of price fluctuations of an asset over time.  
It captures the risk or uncertainty associated with the asset's returns and is a fundamental component in quantitative finance.

**Why we compute volatility features:**

- **Risk Assessment:** High volatility signals greater uncertainty and potential drawdowns, while low volatility implies more stable price behavior.  
- **Strategy Design:** Volatility is used for position sizing, risk-adjusted returns, and adaptive strategies.  
- **Cross-Asset Analysis:** Comparing volatility across assets helps detect market regimes and relative risk levels.  
- **Feature Engineering:** Volatility features serve as inputs for allocation models, regime detection, and systemic market analysis.


## 5.1 Volatility Level

Annualized volatility provides a long-term estimate of risk,
but financial markets do not exhibit constant volatility.

Instead, volatility tends to cluster through time,
a phenomenon known as volatility clustering.

Volatility level measures how risk evolves dynamically
by computing volatility over a moving time window.

Mathematically:

$$
\sigma_t = Std(r_{t-n}, ..., r_t)
$$

Where:

- $n$ is the rolling window length
- $r_t$ represents asset returns

The rolling volatility is then annualized:

$$
\sigma_{annual,t} = \sigma_t \times \sqrt{252}
$$

This metric allows us to identify:

- Risk expansion periods
- Market stress events
- Volatility regimes
- Cross-asset risk transmission

Rising volatility often precedes or accompanies market drawdowns,
making it a key indicator for regime detection.

In [ ]:
# ============================================================
# 5.1 Volatility Level Computation (Annualized)
# ============================================================

# Factor para anualizar volatilidad diaria
ANNUALIZATION_FACTOR = np.sqrt(252)

# Crear un diccionario para guardar las features
vol_level = {}

for window in LOOKBACK_WINDOWS:
    feature_name = f"VOL_{window}"
    
    # Rolling std de retornos diarios
    vol_window = df_returns.rolling(window=window).std()
    
    # Anualizar
    vol_window = vol_window * ANNUALIZATION_FACTOR
    
    # Guardar en diccionario
    vol_level[feature_name] = vol_window

# Concatenar todas las features en un solo DataFrame (MultiIndex: feature x asset)
volatility_level = pd.concat(vol_level, axis=1)
volatility_level.columns.names = ["feature", "asset"]

print("Volatility level:", volatility_level.shape)
display(volatility_level.head())

## 5.2 Volatility Dynamics (VEL, VEL_S, ACC, ACC_S)

While the volatility level (`VOL_N`) measures the magnitude of price fluctuations over a given horizon, it does not capture how volatility itself evolves through time.

To better understand volatility behavior, we compute **volatility dynamics**, which describe the **rate and acceleration of change in volatility**.

### Velocity (VEL)

**Volatility Velocity (`VOL_N_VEL`)** measures the **first derivative of volatility**, i.e., the day-to-day change in volatility:

\[
VEL_t = VOL_t - VOL_{t-1}
\]

This feature captures whether volatility is **increasing or decreasing**.  
Rising velocity may signal **volatility expansion**, while negative values indicate **volatility compression**.

### Smoothed Velocity (VEL_S)

Financial time series are often noisy. To reduce short-term fluctuations, we compute a **smoothed velocity (`VOL_N_VEL_S`)** using a short rolling average.

This helps identify **persistent volatility trends** rather than transient spikes.

### Acceleration (ACC)

**Volatility Acceleration (`VOL_N_ACC`)** represents the **second derivative of volatility**, computed as the change in velocity:

\[
ACC_t = VEL_t - VEL_{t-1}
\]

Acceleration captures **how quickly volatility dynamics themselves are changing**, which can signal **early shifts in market regimes**.

### Smoothed Acceleration (ACC_S)

A smoothed version (`VOL_N_ACC_S`) is also computed to reduce noise and highlight more stable structural changes in volatility dynamics.

---

### Interpretation

These features provide complementary information about volatility:

| Feature | Interpretation |
|-------|------|
| `VOL_N` | Absolute volatility level |
| `VOL_N_VEL` | Change in volatility (expansion vs compression) |
| `VOL_N_VEL_S` | Smoothed volatility trend |
| `VOL_N_ACC` | Change in volatility velocity |
| `VOL_N_ACC_S` | Smoothed structural shifts in volatility dynamics |

Together, these features allow the model to capture **not only how volatile an asset is, but also how volatility is evolving through time**.

In [ ]:
## ==============================================================================##
# This function will be moved to src, after positive diagnostic
## ==============================================================================##
def compute_derivatives(panel: pd.DataFrame, smooth_windows: dict) -> pd.DataFrame:
    """
    Compute VEL, VEL_S, ACC, ACC_S derivatives for a feature panel.

    The function automatically detects base features like:
    VOL_21, MOM_63, etc.

    Parameters
    ----------
    panel : pd.DataFrame
        MultiIndex columns (feature, asset)
    smooth_windows : dict
        Mapping lookback -> smoothing window

    Returns
    -------
    derivatives_panel : pd.DataFrame
    """

    derivatives = {}

    base_features = panel.columns.get_level_values("feature").unique()

    for feat in base_features:

        # detectar prefix y ventana
        prefix, window = feat.split("_")
        window = int(window)

        base = panel[feat]

        smooth = smooth_windows.get(window, 3)

        vel = base.diff()
        vel_s = vel.rolling(smooth).mean()

        acc = vel.diff()
        acc_s = acc.rolling(smooth).mean()

        derivatives[f"{prefix}_{window}_VEL"] = vel
        derivatives[f"{prefix}_{window}_VEL_S"] = vel_s
        derivatives[f"{prefix}_{window}_ACC"] = acc
        derivatives[f"{prefix}_{window}_ACC_S"] = acc_s

    derivatives_panel = pd.concat(derivatives, axis=1)
    derivatives_panel.columns.names = ["feature", "asset"]

    return derivatives_panel

In [ ]:
volatility_dynamics = compute_derivatives(
    panel=volatility_level,
    smooth_windows=SMOOTH_WINDOWS
)
print("volatility_dynamics:", volatility_dynamics.shape)
volatility_dynamics.head()

## 5.3 Volatility Structure

Beyond the level and short-term dynamics of volatility, it is also useful to examine **how volatility behaves across different time horizons**.

Volatility measured over multiple lookback windows captures distinct aspects of market behavior:

- **Short-term volatility** reflects immediate market uncertainty and rapid reactions to recent shocks.
- **Medium-term volatility** captures more persistent periods of turbulence or stabilization.
- **Long-term volatility** reflects the broader structural risk environment of the market.

The relationship between these horizons provides insight into the **shape of the volatility term structure**.

For example:

- When short-term volatility rises relative to long-term volatility, it may signal **recent stress or sudden market shocks**.
- When long-term volatility dominates, it can indicate a **structurally elevated risk environment**.
- Convergence between horizons may suggest **volatility normalization**.

By analyzing the **relative positioning of volatility across horizons**, these features help capture structural changes in risk conditions that may not be visible when looking at a single volatility measure.

### 5.3.1 Volatility Horizon Spreads

To quantify the structure of volatility across horizons, we compute **volatility spreads between different lookback windows**.

These features compare volatility measured at different time horizons and help identify **relative shifts in the volatility term structure**.

Formally, the volatility spread between two horizons is defined as:

$$
VOL\_TERM_{a,b,t} = VOL_{a,t} - VOL_{b,t}
$$

where:

- $a$ represents the **shorter horizon**
- $b$ represents the **longer horizon**
- $t$ denotes time

### Interpretation

Volatility spreads help identify whether **short-term or long-term risk dominates the market environment**.

| Feature | Interpretation |
|-------|------|
| `VOL_TERM_21_63` | Short vs medium volatility pressure |
| `VOL_TERM_21_126` | Short vs longer-term volatility divergence |
| `VOL_TERM_63_252` | Medium vs structural risk environment |

Typical interpretations include:

- **Positive spread** ($VOL_{a,t} > VOL_{b,t}$):  
  Short-term volatility exceeds longer-term volatility, often reflecting **recent shocks or temporary market stress**.

- **Negative spread** ($VOL_{a,t} < VOL_{b,t}$):  
  Longer-term volatility dominates, which may signal a **persistent or structural risk regime**.

- **Spread compression** ($VOL_{a,t} \approx VOL_{b,t}$):  
  Convergence between horizons can indicate **volatility normalization across time scales**.

These features allow the model to capture **the relative shape of the volatility curve**, which often contains more structural information than the absolute volatility level alone.

In [ ]:
# ============================================================
# Volatility Term Structure
# ============================================================

term_features = {}

windows = LOOKBACK_WINDOWS

for i in range(len(windows)):
    for j in range(i + 1, len(windows)):

        short_w = windows[i]
        long_w = windows[j]

        short_label = f"VOL_{short_w}"
        long_label = f"VOL_{long_w}"

        term_label = f"VOL_TERM_{short_w}_{long_w}"

        term_spread = volatility_level[short_label] - volatility_level[long_label]

        term_features[term_label] = term_spread

# Build panel
volatility_term = pd.concat(term_features, axis=1)

# Assign proper MultiIndex names
volatility_term.columns.names = ["feature", "asset"]

print("Volatility term structure shape:", volatility_term.shape)

display(volatility_term.head())

### 5.3.2 Volatility Ratios

In addition to volatility spreads, another way to analyze the structure of volatility across horizons is through **volatility ratios**.

While spreads measure the **absolute difference** between volatility levels at different horizons, ratios measure their **relative magnitude**. This makes them particularly useful in **cross-asset analysis**, where assets can have very different volatility scales.

Formally, the volatility ratio between two horizons is defined as:

$$
VOL\_RATIO_{a,b,t} = \frac{VOL_{a,t}}{VOL_{b,t}}
$$

where:

- $a$ represents the **shorter horizon**
- $b$ represents the **longer horizon**
- $t$ denotes time

### Interpretation

Volatility ratios capture how **short-term risk compares to longer-term risk**.

| Feature | Interpretation |
|-------|------|
| `VOL_RATIO_21_63` | short vs medium-term volatility |
| `VOL_RATIO_21_126` | short vs longer-term volatility |
| `VOL_RATIO_63_252` | medium vs structural volatility |

Typical interpretations include:

- **Ratio > 1**  
  Short-term volatility exceeds longer-term volatility, often indicating **recent market stress or shocks**.

- **Ratio < 1**  
  Longer-term volatility dominates, suggesting a **persistent or structural risk regime**.

- **Ratio ≈ 1**  
  Volatility is similar across horizons, indicating a **stable volatility structure**.

Because ratios are **scale-invariant**, they are often more suitable for models that compare assets with very different volatility levels (e.g., cryptocurrencies vs equities).

In [ ]:
# ============================================================
# Volatility Ratios
# ============================================================

ratio_features = {}

windows = LOOKBACK_WINDOWS

for i in range(len(windows)):
    for j in range(i + 1, len(windows)):

        short_w = windows[i]
        long_w = windows[j]

        short_label = f"VOL_{short_w}"
        long_label = f"VOL_{long_w}"

        ratio_label = f"VOL_RATIO_{short_w}_{long_w}"

        ratio = volatility_level[short_label] / volatility_level[long_label]

        ratio_features[ratio_label] = ratio

# Build panel
volatility_ratio = pd.concat(ratio_features, axis=1)

# Assign proper MultiIndex names
volatility_ratio.columns.names = ["feature", "asset"]

print("Volatility ratio structure shape:", volatility_ratio.shape)

display(volatility_ratio.head())

In [ ]:
# ============================================================
# Build Volatility Structure Panel
# ============================================================

volatility_structure = pd.concat(
    [
        volatility_ratio,
        volatility_term
    ],
    axis=1
)

# Ensure consistent column naming
volatility_structure.columns.names = ["feature", "asset"]

print("volatility_structure panel shape:", volatility_structure.shape)

display(volatility_structure.head())

## 5.4 Volatility of Volatility

While volatility measures the magnitude of market fluctuations, it is also useful to understand **how stable volatility itself is over time**.

This concept is known as **Volatility of Volatility (VoV)**.

VoV measures the dispersion of the volatility process, capturing periods where volatility becomes **erratic or unstable**, which often occurs during **market regime transitions**.

Formally, Volatility of Volatility is defined as the rolling standard deviation of a volatility series:

$$
VOV_{a,b,t} = \sigma \left( VOL_{a,t-i} \right), \quad i = 1,...,b
$$

where:

- $VOL_a$ is the volatility computed using lookback window $a$
- $b$ is the rolling window used to measure the variability of volatility
- $\sigma(\cdot)$ denotes the standard deviation

In expanded form:

$$
VOV_{a,b,t} =
\sqrt{
\frac{1}{b}
\sum_{i=1}^{b}
\left(
VOL_{a,t-i} - \overline{VOL}_{a,t}
\right)^2
}
$$

### Interpretation

Volatility of Volatility captures the **stability of the volatility regime**.

| Feature | Interpretation |
|-------|------|
| `VOV_21_63` | instability of short-term volatility |
| `VOV_63_126` | instability of medium-term volatility |
| `VOV_21_126` | short volatility instability over longer horizon |

Typical interpretations include:

- **High VoV**  
  Volatility is unstable and rapidly changing, often signaling **regime transitions**.

- **Low VoV**  
  Volatility is stable, suggesting a **persistent risk regime**.

In [ ]:
# ============================================================
# Volatility of Volatility
# ============================================================

vov_features = {}

for label, (vol_window, vov_window) in VOV_CONFIG.items():  # vov_config defined in Configuration

    vol_label = f"VOL_{vol_window}"

    vol_series = volatility_level[vol_label]

    vov = vol_series.rolling(window=vov_window).std()

    vov_features[label] = vov

# Build panel
volatility_of_volatility = pd.concat(vov_features, axis=1)

volatility_of_volatility.columns.names = ["feature", "asset"]

print("Volatility of volatility shape:", volatility_of_volatility.shape)

display(volatility_of_volatility.head())

## 5.5.1 Volatility Regime Indicators

Raw volatility levels are difficult to interpret across time because financial markets naturally move through different volatility environments.

For example, a volatility level of 20% may represent **extreme stress in one regime** but **normal conditions in another**, depending on the historical context.

To make volatility more comparable through time, we construct **Volatility Regime Indicators** based on normalization and relative positioning.

These indicators aim to answer questions such as:

- Is volatility currently high relative to its historical distribution?
- Is volatility expanding or compressing relative to its past behavior?
- Is the market entering a high-risk regime?

To capture these dynamics, we construct three complementary indicators:

1. **Volatility Z-Score** — measures how many standard deviations current volatility is from its historical mean.

2. **Volatility Percentile** — measures the relative rank of current volatility within its historical distribution.

3. **Volatility Expansion / Compression** — identifies whether volatility is increasing or decreasing relative to its recent history.

These indicators transform raw volatility into **regime-aware signals**, which are often more informative for models focused on **risk management, asset allocation, and regime detection**.

#### Volatility Z-Score

The Volatility Z-Score measures how extreme the current volatility level is relative to its historical distribution.

Formally:

$$
ZVOL_{w,t} =
\frac{
VOL_{w,t} - \mu_{w,t}
}{
\sigma_{w,t}
}
$$

where:

- $VOL_{w,t}$ is the volatility computed using lookback window $w$
- $\mu_{w,t}$ is the rolling mean of the volatility series
- $\sigma_{w,t}$ is the rolling standard deviation of the volatility series

### Interpretation

| Z-Score | Interpretation |
|------|------|
| > 2 | extremely high volatility |
| 1 to 2 | elevated volatility |
| -1 to 1 | normal regime |
| < -1 | unusually low volatility |

Z-scores are widely used in quantitative models because they **normalize features across time**, making them more comparable across different volatility regimes.

In [ ]:
# ============================================================
# Volatility Z-Score
# ============================================================

zvol_features = {}

z_window = Z_SCORE_WINDOW # defined in Configuration

for window in LOOKBACK_WINDOWS:

    vol_label = f"VOL_{window}"
    z_label = f"VOL_{window}_Z"

    vol_series = volatility_level[vol_label]

    rolling_mean = vol_series.rolling(z_window).mean()
    rolling_std = vol_series.rolling(z_window).std()

    zscore = (vol_series - rolling_mean) / rolling_std

    zvol_features[z_label] = zscore

# Build panel
volatility_zscore = pd.concat(zvol_features, axis=1)

volatility_zscore.columns.names = ["feature", "asset"]

print("Volatility Z-Score shape:", volatility_zscore.shape)

display(volatility_zscore.head())

### 5.5.2 Volatility Percentile

While Z-scores measure how many standard deviations the current volatility is from its mean, another useful normalization approach is to compute the **percentile rank of volatility within its historical distribution**.

The Volatility Percentile measures the **relative position of the current volatility level within a rolling historical window**.

Formally, the percentile rank can be expressed as:

$$
VPERC_{w,t} =
\frac{
\#\{VOL_{w,i} \leq VOL_{w,t}\}
}{
N
}
$$

where:

- $VOL_{w,t}$ is the volatility computed with lookback window $w$
- $N$ is the number of observations in the rolling window
- $\#\{\cdot\}$ counts the number of past observations less than or equal to the current value

The resulting value lies between:

$$
0 \le VPERC_{w,t} \le 1
$$

### Interpretation

| Percentile | Interpretation |
|------|------|
| > 0.9 | extremely high volatility regime |
| 0.7 – 0.9 | elevated volatility |
| 0.3 – 0.7 | normal regime |
| < 0.3 | low volatility regime |

Percentiles are particularly useful because they **do not assume normality of the volatility distribution**, making them robust for financial time series where volatility often exhibits skewness and heavy tails.

In [ ]:
# ============================================================
# Volatility Percentile (optimized)
# ============================================================

vperc_features = {}

window = PERCENTILE_WINDOW

for vol_window in LOOKBACK_WINDOWS:

    vol_label = f"VOL_{vol_window}"
    perc_label = f"VOL_{vol_window}_P"

    vol_df = volatility_level[vol_label]  # dataframe: date x asset

    percentile_df = pd.DataFrame(index=vol_df.index, columns=vol_df.columns)

    for asset in vol_df.columns:

        s = vol_df[asset]

        percentile = (
            s.rolling(window)
            .apply(lambda x: (x <= x.iloc[-1]).mean(), raw=False)
        )

        percentile_df[asset] = percentile

    vperc_features[perc_label] = percentile_df

# Build panel
volatility_percentile = pd.concat(vperc_features, axis=1)
volatility_percentile.columns.names = ["feature", "asset"]

print("Volatility percentile shape:", volatility_percentile.shape)

display(volatility_percentile.head())

### 5.5.3 Volatility Expansion and Compression

Changes in volatility regimes are often preceded by periods of **volatility compression**, followed by sudden **volatility expansion** when market stress increases.

To capture these dynamics, we compare **short-term volatility** with **medium-term volatility**. When short-term volatility rises relative to the medium-term level, it indicates that market uncertainty is increasing.

Formally, the Volatility Expansion indicator is defined as:

$$
VOL\_EXP_t =
\frac{VOL_{21,t}}{VOL_{63,t}} - 1
$$

where:

- $VOL_{21,t}$ represents short-term volatility (21-day window)
- $VOL_{63,t}$ represents medium-term volatility (63-day window)

### Interpretation

| Value | Interpretation |
|------|---------------|
| $> 0$ | volatility expansion |
| $\approx 0$ | stable volatility regime |
| $< 0$ | volatility compression |

Positive values indicate that **short-term volatility exceeds medium-term volatility**, suggesting that market risk is **increasing**.  
Negative values indicate that **short-term volatility is lower than medium-term volatility**, suggesting **volatility compression** and more stable market conditions.

This indicator is particularly useful for detecting **early transitions into high-volatility regimes**.

In [ ]:
# ============================================================
# Volatility Expansion / Compression
# ============================================================

exp_features = {}

for short_w, long_w in EXPANSION_PAIRS:

    short_label = f"VOL_{short_w}"
    long_label  = f"VOL_{long_w}"

    feature_name = f"VOL_EXP_{short_w}_{long_w}"

    vol_exp = (
        volatility_level[short_label] /
        volatility_level[long_label]
    ) - 1

    exp_features[feature_name] = vol_exp


volatility_expansion = pd.concat(exp_features, axis=1)

volatility_expansion.columns.names = ["feature", "asset"]

print("Volatility expansion shape:", volatility_expansion.shape)

display(volatility_expansion.head())

## 5.6 Build Volatility Feature Panel

After computing the different volatility feature groups, we combine them into a single **volatility feature panel**.

Each feature family captures a different aspect of volatility dynamics:

- **Volatility Level**: absolute volatility estimates across horizons
- **Volatility Dynamics**: changes in volatility (velocity, acceleration)
- **Volatility Structure**: relationships between volatility horizons
- **Volatility of Volatility**: variability of volatility itself
- **Volatility Regime Indicators**: indicators of volatility regimes

All feature groups share the same structure:

- index: `date`
- columns: `MultiIndex(feature, asset)`

This allows us to concatenate them along the **feature dimension** to build a unified volatility feature dataset.

In [ ]:
# ============================================================
# Build Volatility Feature Panel
# ============================================================

volatility_features = pd.concat(
    [
        volatility_level,
        volatility_dynamics,
        volatility_structure,
        volatility_of_volatility,
        volatility_zscore,
        volatility_percentile,
        volatility_expansion,
    ],
    axis=1
)

# Ensure consistent column naming
volatility_features.columns.names = ["feature", "asset"]

print("Volatility feature panel shape:", volatility_features.shape)

display(volatility_features.head())

In [ ]:
volatility_features.columns.get_level_values("asset").unique()

# 6. Diagnosis/Validation

## 6.1 Dataset integrity checks

In [ ]:
# ==========================================
# 6.1 DATASET INTEGRITY CHECKS
# ==========================================

features_panel = volatility_features

print("===== DATASET DIMENSIONS =====")

# Rows correspond to time observations (dates)
# Columns correspond to feature × asset combinations

print("Rows (dates):", features_panel.shape[0])
print("Columns (feature × asset):", features_panel.shape[1])

print()


print("===== MULTIINDEX STRUCTURE =====")

# Validate MultiIndex structure

print("Column levels:", features_panel.columns.names)

n_features = features_panel.columns.get_level_values("feature").nunique()
n_assets = features_panel.columns.get_level_values("asset").nunique()

print("Number of features:", n_features)
print("Number of assets:", n_assets)

print()


print("===== FEATURE LIST =====")

# List of computed features

features = features_panel.columns.get_level_values("feature").unique()

print(features)

print()


print("===== ASSET LIST =====")

assets = features_panel.columns.get_level_values("asset").unique()

print(assets)

print()


print("===== MISSING DATA CHECK =====")

# Rolling windows naturally generate NaNs at the beginning

nan_counts = features_panel.isna().sum().sum()
nan_ratio = features_panel.isna().mean().mean()

print("Total NaN values:", nan_counts)
print("Average NaN ratio:", round(nan_ratio, 4))

print()


print("===== DATE RANGE =====")

print("Start date:", features_panel.index.min())
print("End date:", features_panel.index.max())

print()


print("===== BASIC STATISTICS =====")

# Inspect summary statistics

display(features_panel.describe().T.head())

## 6.2 Feature Diagnostics

Beyond structural validation, we analyze the statistical behavior of the
constructed features.

These diagnostics help identify:

- degenerate signals (no cross-sectional variation)
- unstable distributions
- redundant features
- highly correlated signals

The following diagnostics are performed:

1. Cross-Sectional Dispersion
2. Cross-Sectional Distribution
3. Feature Correlation
4. Feature Clustering

### 6.2.1 Cross-Sectional Signal Disperion

### 6.2.1 Cross-Sectional Signal Dispersion

We evaluate whether each feature meaningfully differentiates assets at a given
point in time.

For each feature and date we compute the cross-sectional standard deviation
across assets.

Higher dispersion indicates stronger differentiation between assets,
which is desirable for cross-sectional models.

In [ ]:
# ==========================================
# 6.2.1 CROSS-SECTIONAL SIGNAL DISPERSION
# ==========================================

print("===== CROSS-SECTIONAL SIGNAL DISPERSION =====")

panel = features_panel

# Transpose for easier feature grouping
panel_t = panel.T  # (feature, asset) x date

# Compute dispersion across assets
cs_dispersion = panel_t.groupby(level="feature").std().T

# Average dispersion through time
avg_dispersion = cs_dispersion.mean().sort_values(ascending=False)

print("\nAverage cross-sectional dispersion per feature:")
print(avg_dispersion)

print("\nTop 10 most dispersed signals:")
print(avg_dispersion.head(10))

print("\nBottom 10 least dispersed signals:")
print(avg_dispersion.tail(10))

### 6.2.1b Normalized Signal Dispersion

To account for differences in signal scale, we compute normalized dispersion.

This metric compares cross-sectional dispersion to the average signal magnitude.

Higher values indicate stronger differentiation relative to signal amplitude.

In [ ]:
# ==========================================
# NORMALIZED SIGNAL DISPERSION
# ==========================================

print("\n===== NORMALIZED SIGNAL DISPERSION =====")

panel = features_panel
panel_t = panel.T

# Average signal magnitude
signal_scale = (
    panel_t
    .abs()
    .groupby(level="feature")
    .mean()
    .mean(axis=1)
)

# Normalized dispersion
normalized_dispersion = (avg_dispersion / signal_scale).sort_values(ascending=False)

print("\nNormalized dispersion per feature:")
print(normalized_dispersion)

print("\nTop 10 normalized signals:")
print(normalized_dispersion.head(10))

print("\nBottom 10 normalized signals:")
print(normalized_dispersion.tail(10))

In [ ]:
signal_diagnostics = pd.DataFrame({
    "dispersion": avg_dispersion,
    "normalized_dispersion": normalized_dispersion
}).sort_values("normalized_dispersion", ascending=False)

display(signal_diagnostics)

### 6.2.2 Cross-Sectional Rank Spread

While cross-sectional dispersion measures the overall variability of a signal
across assets, it does not necessarily indicate whether the signal effectively
separates the highest and lowest ranked assets.

To evaluate the ranking power of each feature, we compute the **cross-sectional
rank spread**.

For each date, assets are ranked according to the signal value. We then
compare the average signal of the highest-ranked assets to the lowest-ranked
assets.

The cross-sectional rank spread is defined as:
 $Spread_{f,t} = E[x_{f,i,t} | i \in Top_q] - E[x_{f,i,t} | i \in Bottom_q]$.

where:

- \( f \) denotes the feature  
- \( i \) denotes the asset  
- \( t \) denotes the time index  
- \( q \) denotes the quantile threshold

In this study we use a **20% quantile threshold**, meaning that the top and
bottom 20% of assets are used to compute the spread.

Interpretation:

• **High spread**

The signal clearly differentiates between top and bottom assets and may be
useful for cross-sectional ranking strategies.

• **Moderate spread**

The signal provides some ranking information but may be weaker.

• **Low spread**

The signal does not meaningfully separate assets and may contain little
cross-sectional information.

In [ ]:
# ==========================================
# 6.2.2 CROSS-SECTIONAL RANK SPREAD
# ==========================================

print("\n===== CROSS-SECTIONAL RANK SPREAD =====")

panel = features_panel

base_q = 0.2

rank_spreads = {}

for feature in panel.columns.get_level_values("feature").unique():

    feature_df = panel[feature]  # date x asset

    n_assets = feature_df.shape[1]

    # number of assets to select
    n_select = max(1, int(np.floor(n_assets * base_q)))

    # effective quantile
    q = n_select / n_assets

    ranks = feature_df.rank(axis=1, pct=True)

    top = feature_df.where(ranks >= 1 - q)
    bottom = feature_df.where(ranks <= q)

    spread = top.mean(axis=1) - bottom.mean(axis=1)

    rank_spreads[feature] = spread.mean()

rank_spreads = pd.Series(rank_spreads).sort_values(ascending=False)
print(f"Assets: {n_assets} | Effective quantile: {q:.2f}")
print(rank_spreads)

In [ ]:
feature_diagnostics = pd.concat([
    avg_dispersion,
    normalized_dispersion,
    rank_spreads
], axis=1)

feature_diagnostics.columns = [
    "dispersion",
    "normalized_dispersion",
    "rank_spread"
]

feature_diagnostics.sort_values(
    "rank_spread",
    ascending=False
)

In [ ]:
fig = px.bar(
    avg_dispersion.sort_values(),
    orientation="h",
    title="Cross-Sectional Signal Dispersion",
    labels={"value": "Dispersion", "index": "Feature"}
)

fig.show()

In [ ]:
import plotly.express as px

fig = px.bar(
    rank_spreads.sort_values(),
    orientation="h",
    title="Cross-Sectional Rank Spread by Feature",
    labels={"value": "Rank Spread", "index": "Feature"}
)

fig.show()

In [ ]:
diagnostics = pd.DataFrame({
    "rank_spread": rank_spreads,
    "signal_dispersion": avg_dispersion
})

fig = px.scatter(
    diagnostics,
    x="rank_spread",
    y="signal_dispersion",
    text=diagnostics.index,
    title="Feature Diagnostic Map"
)

fig.update_traces(
    textposition="top center",
    textfont=dict(size=7)
)

fig.show()

In [ ]:
top_features = diagnostics.nlargest(10, "rank_spread").index

diagnostics["label"] = diagnostics.index.where(
    diagnostics.index.isin(top_features),
    ""
)

fig = px.scatter(
    diagnostics,
    x="rank_spread",
    y="signal_dispersion",
    text="label",
    title="Feature Diagnostic Map"
)

fig.update_traces(textposition="top center", textfont=dict(size=9))

fig.add_vline(x=diagnostics.rank_spread.median(), line_dash="dash")
fig.add_hline(y=diagnostics.signal_dispersion.median(), line_dash="dash")

fig.show()

In [ ]:
def classify_feature(name):

    if "VEL" in name or "ACC" in name:
        return "VOL_DERIVATIVES"

    if "VOV" in name:
        return "VOL_OF_VOL"

    if "TERM" in name or "RATIO" in name or "EXP" in name:
        return "VOL_TERM_STRUCTURE"

    if "_Z" in name or "_P" in name:
        return "VOL_NORMALIZED"

    if name.startswith("VOL_"):
        return "VOL_LEVEL"

    return "OTHER"


diagnostics["feature_type"] = diagnostics.index.map(classify_feature)

In [ ]:
fig = px.scatter(
    diagnostics,
    x="rank_spread",
    y="signal_dispersion",
    color="feature_type",
    hover_name=diagnostics.index,
    title="Feature Diagnostic Map",
    width=900,
    height=600
)

top_features = diagnostics.nlargest(8, "rank_spread").index

diagnostics["label"] = diagnostics.index.where(
    diagnostics.index.isin(top_features),
    ""
)

fig.update_traces(
    text=diagnostics["label"],
    textposition="top center",
    textfont=dict(size=9)
)

fig.add_vline(
    x=diagnostics.rank_spread.median(),
    line_dash="dash"
)

fig.add_hline(
    y=diagnostics.signal_dispersion.median(),
    line_dash="dash"
)

fig.show()

### 6.2.3 Feature Correlation Analysis

In this section we analyze the **correlation between features** to detect:

- **Signal redundancy**
- **Structural clusters of indicators**
- **Effective dimensionality of the feature space**

The correlation between two features $f_i$ and $f_j$ is defined as:

$$
\rho_{ij} =
\frac{\text{Cov}(f_i, f_j)}
{\sigma_i \sigma_j}
$$

where:

- $\text{Cov}(f_i,f_j)$ is the covariance between the two signals  
- $\sigma_i, \sigma_j$ are their standard deviations

Interpretation:

| Correlation | Interpretation |
|---|---|
| $ \rho \approx 1 $ | highly redundant signals |
| $ \rho \approx 0 $ | independent signals |
| $ \rho \approx -1 $ | inversely related signals |

In **quantitative feature engineering** we aim for:

- a **diverse set of signals**
- **low redundancy**

To compute correlations we first transform the feature panel into a format where:

- each **row** represents an observation $(date, asset)$
- each **column** represents a **feature**

We then compute the **feature correlation matrix** and visualize it using a **heatmap**.

This analysis typically reveals **natural clusters of indicators**, for example:

- volatility level
- normalized volatility
- volatility term structure
- volatility derivatives
- volatility-of-volatility

These clusters help us understand the **structure of the feature space** and identify indicators that provide **genuinely new information**, as opposed to those that are simply **variations of the same signal**.

In [ ]:
# ==========================================
# 6.2.3 FEATURE CORRELATION ANALYSIS
# ==========================================

print("\n===== FEATURE CORRELATION ANALYSIS =====")

panel = features_panel

# ------------------------------------------
# Convert panel to long format
# (date, asset) x feature
# ------------------------------------------

panel_long = panel.stack(level="asset")

print("Panel long shape:", panel_long.shape)

# ------------------------------------------
# Compute correlation matrix
# ------------------------------------------

feature_corr = panel_long.corr()

print("Correlation matrix shape:", feature_corr.shape)

# ------------------------------------------
# Plot heatmap
# ------------------------------------------

import plotly.express as px

fig = px.imshow(
    feature_corr,
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="Feature Correlation Matrix"
)

fig.update_layout(
    width=900,
    height=900
)

fig.show()

### 6.2.4 Hierarchical Feature Clustering

While the raw correlation matrix provides useful information, it is often difficult to visually identify **groups of related features** when the matrix is ordered arbitrarily.

To better reveal the structure of the feature space, we apply **hierarchical clustering** to the correlation matrix.

The idea is to group features based on their **similarity**, measured through their pairwise correlations.

A common transformation is to convert correlation into a **distance metric**:

$$
d_{ij} = 1 - |\rho_{ij}|
$$

where:

- $\rho_{ij}$ is the correlation between features $i$ and $j$
- $d_{ij}$ represents the distance between the two signals

This distance metric ensures that:

- highly correlated features have **small distances**
- weakly correlated features have **larger distances**

Using this distance matrix, hierarchical clustering builds a **tree structure (dendrogram)** that groups similar features together.

The reordered correlation matrix reveals **natural clusters of signals**, which often correspond to feature families such as:

- volatility level indicators
- normalized volatility signals
- volatility term structure features
- volatility derivatives
- volatility-of-volatility measures

This step is extremely useful for:

- detecting **redundant signals**
- identifying **feature families**
- guiding **feature selection**
- reducing the **effective dimensionality** of the feature space

In [ ]:
# ==========================================
# 6.2.4 HIERARCHICAL FEATURE CLUSTERING
# ==========================================

print("\n===== HIERARCHICAL FEATURE CLUSTERING =====")

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform

# ------------------------------------------
# Convert correlation to distance matrix
# ------------------------------------------

distance_matrix = 1 - np.abs(feature_corr)

# linkage requires condensed distance matrix
condensed_dist = squareform(distance_matrix)

linkage_matrix = linkage(condensed_dist, method="average")

# ------------------------------------------
# Clustered heatmap
# ------------------------------------------

sns.clustermap(
    feature_corr,
    row_linkage=linkage_matrix,
    col_linkage=linkage_matrix,
    cmap="RdBu_r",
    center=0,
    figsize=(12,12)
)

plt.title("Hierarchically Clustered Feature Correlation Matrix")
plt.show()

### 6.2.5 Feature Redundancy Detection

In large feature spaces, some indicators may carry **almost identical information**, i.e., they are **highly correlated** with each other.

Detecting these redundant features helps:

- **Reduce dimensionality**
- **Avoid multicollinearity**
- **Simplify feature selection**
- **Focus on truly informative signals**

We define redundancy as:

$$
|\rho_{ij}| > \rho_{threshold}
$$

where $\rho_{ij}$ is the correlation between features $i$ and $j$, and $\rho_{threshold}$ is typically set to 0.95–0.99.

This process outputs:

- a **list of highly correlated feature pairs**
- optionally, a **suggestion of features to drop** to remove redundancy

By systematically removing or combining redundant features, the **effective dimensionality** of the feature space is reduced without significant loss of information.

In [ ]:
# ==========================================
# 6.2.5 Feature Redundancy Detection
# ==========================================

print("\n===== FEATURE REDUNDANCY DETECTION =====")

# Threshold for considering features redundant
redundancy_threshold = 0.95

# Compute absolute correlation matrix
abs_corr = feature_corr.abs()

# Upper triangle mask to avoid duplicate pairs
upper_tri = np.triu(np.ones_like(abs_corr, dtype=bool), k=1)

# Select pairs with correlation above threshold
redundant_pairs = [
    (abs_corr.index[i], abs_corr.columns[j], abs_corr.iloc[i, j])
    for i in range(abs_corr.shape[0])
    for j in range(abs_corr.shape[1])
    if upper_tri[i, j] and abs_corr.iloc[i, j] > redundancy_threshold
]

# Convert to DataFrame for inspection
redundant_df = pd.DataFrame(redundant_pairs, columns=["Feature_1", "Feature_2", "Correlation"])
redundant_df = redundant_df.sort_values(by="Correlation", ascending=False).reset_index(drop=True)

print(f"Number of highly correlated feature pairs (>|{redundancy_threshold}|): {len(redundant_df)}")
display(redundant_df.head(20))  # show top 20 for quick inspection

### 6.2.6 Cross-Sectional Distribution Diagnostics

Analyzing the **cross-sectional distribution** of features at each time step helps us identify:

- Outliers in the feature space
- Skewness or heavy tails
- Differences in dispersion across features
- Features that may require normalization or winsorization before modeling

For a given feature \(f\) and time \(t\), let \(x_{t,i}^f\) denote the value for asset \(i\). We can compute:

1. **Cross-sectional mean**:
$$
\mu_t^f = \frac{1}{N} \sum_{i=1}^{N} x_{t,i}^f
$$

2. **Cross-sectional standard deviation**:
$$
\sigma_t^f = \sqrt{\frac{1}{N} \sum_{i=1}^{N} \left(x_{t,i}^f - \mu_t^f \right)^2}
$$

3. **Cross-sectional skewness** (optional):
$$
\text{skew}_t^f = \frac{\frac{1}{N} \sum_i (x_{t,i}^f - \mu_t^f)^3}{(\sigma_t^f)^3}
$$

We summarize the distribution **over time** by computing:

- Average cross-sectional mean
- Average cross-sectional standard deviation
- Extreme values or outliers

This diagnostic helps **compare features** and ensures that signals are **well-behaved across assets**, ready for factor modeling or cross-asset allocation.

In [ ]:
# ==========================================
# 6.2.6 Cross-Sectional Distribution Diagnostics (with Skewness)
# ==========================================

import pandas as pd
import plotly.express as px

print("\n===== CROSS-SECTIONAL DISTRIBUTION DIAGNOSTICS WITH SKEWNESS =====")

panel = features_panel  # wide panel: dates x (feature x asset)
panel_t = panel.T       # (feature, asset) x dates

features = panel_t.index.get_level_values("feature").unique()

# Crear DataFrame para stats
cs_distribution = pd.DataFrame(index=features, columns=[
    "cross_sectional_mean",
    "cross_sectional_std",
    "cross_sectional_skew",
    "cross_sectional_max",
    "cross_sectional_min"
], dtype=float)

# Calcular estadísticos
for f in features:
    feat_df = panel_t.xs(f)  # fechas x assets
    mu = feat_df.mean(axis=1)
    sigma = feat_df.std(axis=1)
    cs_distribution.loc[f, "cross_sectional_mean"] = mu.mean()
    cs_distribution.loc[f, "cross_sectional_std"] = sigma.mean()
    cs_distribution.loc[f, "cross_sectional_skew"] = ((feat_df.sub(mu, axis=0)**3).mean(axis=1) / sigma**3).mean()
    cs_distribution.loc[f, "cross_sectional_max"] = feat_df.max(axis=1).max()
    cs_distribution.loc[f, "cross_sectional_min"] = feat_df.min(axis=1).min()

cs_distribution = cs_distribution.sort_values(by="cross_sectional_std", ascending=False)

print("Cross-sectional distribution summary (with skewness):")
display(cs_distribution)

# Plot histogram para features seleccionadas
sample_features = cs_distribution.index[:8]  # top 8 por std

for feat in sample_features:
    fig = px.histogram(
        panel[feat],
        nbins=20,
        title=f"Cross-Sectional Distribution: {feat}"
    )
    fig.show()

In [ ]:
import plotly.express as px

# Prepara el dataframe de diagnóstico
cs_diagnostics = cs_distribution.copy()  # tu tabla con mean, std, skew, max, min

# Scatter plot: dispersion (std) vs skewness
fig = px.scatter(
    cs_diagnostics,
    x="cross_sectional_std",
    y="cross_sectional_skew",
    text=cs_diagnostics.index,
    title="Cross-Sectional Feature Diagnostic Map",
    labels={
        "cross_sectional_std": "Cross-Sectional Std (Dispersion)",
        "cross_sectional_skew": "Cross-Sectional Skewness"
    },
    height=600,
    width=800
)

# Ajustes de texto y marcadores
fig.update_traces(
    textposition="top center",
    marker=dict(size=7, color='royalblue', opacity=0.8)
)

fig.show()

# 7. Visualization

In [ ]:
# ============================================================
# 7. VISUALIZATION / EXPLORATORY PLOTS
# ============================================================

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 7.1 Configuration for plots
# ------------------------------------------------------------
SAMPLE_ASSETS = ["BTC", "SPY"]          # sample assets to inspect
SAMPLE_FEATURES = ["VOL_252_Z", "VOL_126_Z", "VOL_21_Z"]  # features to inspect
TEXT_POSITION = "top center"
FIGURE_WIDTH = 1000
FIGURE_HEIGHT = 600

# ------------------------------------------------------------
# 7.2 Single Feature Across All Assets (with Prices)
# ------------------------------------------------------------
for feature in SAMPLE_FEATURES:
    df_feature = features_panel.xs(feature, level="feature", axis=1)
    
    # Crear figura con doble eje
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    # Agregar features
    for asset in df_feature.columns:
        fig.add_trace(
            go.Scatter(
                x=df_feature.index,
                y=df_feature[asset],
                mode="lines",
                name=f"{asset} Feature"
            ),
            secondary_y=False
        )
    
    # Agregar precios de los activos en eje secundario
    for asset in SAMPLE_ASSETS:
        fig.add_trace(
            go.Scatter(
                x=df_prices.index,
                y=df_prices[asset],
                mode="lines",
                line=dict(dash="dot"),
                name=f"{asset} Price"
            ),
            secondary_y=True
        )
    
    # Layout
    fig.update_layout(
        title=f"{feature} Across Assets with Prices",
        xaxis_title="Date",
        width=FIGURE_WIDTH,
        height=FIGURE_HEIGHT
    )
    fig.update_yaxes(title_text="Feature Value", secondary_y=False)
    fig.update_yaxes(title_text="Price", secondary_y=True)
    
    fig.show()

# ------------------------------------------------------------
# 7.3 Single Asset Across Multiple Features (with Prices)
# ------------------------------------------------------------
for asset in SAMPLE_ASSETS:
    df_asset = features_panel.xs(asset, level="asset", axis=1)
    df_asset = df_asset[SAMPLE_FEATURES]
    
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    # Features
    for feature in df_asset.columns:
        fig.add_trace(
            go.Scatter(
                x=df_asset.index,
                y=df_asset[feature],
                mode="lines",
                name=f"{feature}"
            ),
            secondary_y=False
        )
    
    # Prices
    fig.add_trace(
        go.Scatter(
            x=df_prices.index,
            y=df_prices[asset],
            mode="lines",
            line=dict(dash="dot"),
            name=f"{asset} Price"
        ),
        secondary_y=True
    )
    
    fig.update_layout(
        title=f"Selected Features for {asset} with Price",
        xaxis_title="Date",
        width=FIGURE_WIDTH,
        height=FIGURE_HEIGHT
    )
    fig.update_yaxes(title_text="Feature Value", secondary_y=False)
    fig.update_yaxes(title_text="Price", secondary_y=True)
    
    fig.show()

# ------------------------------------------------------------
# 7.4 Feature Diagnostic Map (Scatter)
# ------------------------------------------------------------
# Asegurate de tener 'diagnostics' dataframe con columnas 'rank_spread' y 'signal_dispersion'
fig = px.scatter(
    diagnostics,
    x="rank_spread",
    y="signal_dispersion",
    text=diagnostics.index,
    title="Feature Diagnostic Map"
)
fig.update_traces(textposition=TEXT_POSITION)
fig.update_layout(width=FIGURE_WIDTH, height=FIGURE_HEIGHT)
fig.show()

# ------------------------------------------------------------
# 7.5 Heatmap of Feature Correlations
# ------------------------------------------------------------
corr_matrix = features_panel.corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.show()

# ------------------------------------------------------------
# 7.6 Outlier Inspection for a Feature (with Prices)
# ------------------------------------------------------------
feature_to_inspect = "VOL_252_Z"
df_feature = features_panel.xs(feature_to_inspect, level="feature", axis=1)
mean_series = df_feature.mean()
std_series = df_feature.std()
upper_bound = mean_series + 3 * std_series
lower_bound = mean_series - 3 * std_series

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Features
for asset in df_feature.columns:
    fig.add_trace(
        go.Scatter(
            x=df_feature.index,
            y=df_feature[asset],
            mode="lines",
            name=f"{asset} Feature"
        ),
        secondary_y=False
    )
    
    # Outliers
    outliers = df_feature[asset][(df_feature[asset] > upper_bound[asset]) | (df_feature[asset] < lower_bound[asset])]
    fig.add_trace(
        go.Scatter(
            x=outliers.index,
            y=outliers.values,
            mode="markers",
            marker=dict(color="red", size=8),
            name=f"{asset} outlier"
        ),
        secondary_y=False
    )

# Prices
for asset in SAMPLE_ASSETS:
    fig.add_trace(
        go.Scatter(
            x=df_prices.index,
            y=df_prices[asset],
            mode="lines",
            line=dict(dash="dot"),
            name=f"{asset} Price"
        ),
        secondary_y=True
    )

fig.update_layout(
    title=f"Outlier Inspection for {feature_to_inspect} with Prices",
    xaxis_title="Date",
    width=FIGURE_WIDTH,
    height=FIGURE_HEIGHT
)
fig.update_yaxes(title_text="Feature Value", secondary_y=False)
fig.update_yaxes(title_text="Price", secondary_y=True)

fig.show()

# 8. Exports

In [ ]:
# -----------------------------------------
# 8.1 Final Feature Panel
# -----------------------------------------

print("Feature panel shape:", features_panel.shape)
print("Column levels:", features_panel.columns.names)

features_panel.head()

In [ ]:
# -----------------------------------------
# Panel metadata
# -----------------------------------------

n_dates = features_panel.shape[0]
n_features = features_panel.columns.get_level_values("feature").nunique()
n_assets = features_panel.columns.get_level_values("asset").nunique()

print("Dates:", n_dates)
print("Features:", n_features)
print("Assets:", n_assets)

In [ ]:
# ============================================================
# 8. EXPORT / SAVE FEATURES
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# 8.1 Helper: detect project root
# ------------------------------------------------------------
def find_project_root(marker_folder="data"):
    current_path = os.path.abspath(os.getcwd())
    while True:
        if marker_folder in os.listdir(current_path):
            return current_path
        parent = os.path.dirname(current_path)
        if parent == current_path:
            raise RuntimeError(f"No '{marker_folder}' folder found in directory tree.")
        current_path = parent

project_root = find_project_root("data")
print("Project root detected:", project_root)

# ------------------------------------------------------------
# 8.2 Feature family configuration
# ------------------------------------------------------------
feature_family = FEATURE_FAMILY  # e.g., "volatility", "momentum"
base_feature_dir = os.path.join(project_root, "data", "features", "asset", feature_family)
os.makedirs(base_feature_dir, exist_ok=True)

# ------------------------------------------------------------
# 8.3 Clean and sort panel
# ------------------------------------------------------------
features_panel = features_panel.sort_index()         # sort rows by date
features_panel = features_panel.sort_index(axis=1)  # sort MultiIndex columns

# ------------------------------------------------------------
# 8.4 Export per asset
# ------------------------------------------------------------
assets = features_panel.columns.get_level_values("asset").unique()

for asset in assets:
    df_asset = features_panel.xs(asset, level="asset", axis=1)
    df_asset.columns.name = None  # clean column metadata
    filename = os.path.join(base_feature_dir, f"{asset}.parquet")
    df_asset.to_parquet(filename, compression="zstd")
    print(f"{feature_family} exported for {asset} -> {filename}")